# 10 Pi Runtime + Latency / Sequence Validation v1

이 노트북은 09에서 만든 ONNX 모델과 07/08 후처리가 **Pi 환경에서도 같은 결과와 충분한 속도로 동작하는지** 확인하는 최종 배포 검증 노트북이다.

여기서는 새 학습, 새 decoder 선택, 새 steering 정책 설계를 하지 않는다.

검증 대상:

1. local ONNX reference와 Pi ONNX raw output이 같은가
2. 07 decoder 적용 후 lane 결과가 같은가
3. 08 steering 적용 후 `mode / steer_norm`이 같은가
4. Pi runtime latency가 어느 정도인가
5. Pi에서 field3 offline replay 영상을 만들 수 있는가

## 실행 모드

같은 노트북을 두 환경에서 실행한다.

### Local / Windows

- `review_outputs/10_pi_runtime_latency_sequence_validation_v1/pkg/` 생성
- ONNX + `.onnx.data` + contracts + sample images 패키징
- local ONNX reference 생성
- local ONNX latency baseline 저장
- optional local PyTorch `.pth` latency baseline 저장

### Raspberry Pi

- local에서 만든 `pkg/` 폴더 전체를 Pi로 복사
- `pkg/` 폴더 안에서 이 노트북 실행
- Pi ONNX output을 local reference와 비교
- Pi latency와 field3 replay video 저장

`pkg/`처럼 짧은 폴더명을 쓰는 이유는 Windows local 준비 단계에서 경로 길이 문제를 피하기 위해서다.

In [1]:
from pathlib import Path
import hashlib
import json
import math
import os
import platform
import shutil
import sys
import time

import numpy as np
import pandas as pd

try:
    import cv2
except ImportError as exc:
    raise ImportError("OpenCV is required. Install opencv-python or opencv-python-headless.") from exc

try:
    import onnxruntime as ort
except ImportError as exc:
    raise ImportError("onnxruntime is required. On Pi, install an ARM-compatible onnxruntime package.") from exc

REQUIRE_SCIPY_FOR_DECODER = True
try:
    from scipy.interpolate import InterpolatedUnivariateSpline
    HAS_SCIPY = True
except ImportError:
    InterpolatedUnivariateSpline = None
    HAS_SCIPY = False

if REQUIRE_SCIPY_FOR_DECODER and not HAS_SCIPY:
    raise ImportError("scipy is required for exact official Lane.to_array-style spline resampling. Install scipy.")

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

print("python:", sys.version)
print("machine:", platform.machine(), "platform:", platform.platform())
print("onnxruntime:", ort.__version__)
print("opencv:", cv2.__version__)
print("scipy spline available:", HAS_SCIPY)

python: 3.10.18 | packaged by Anaconda, Inc. | (main, Jun  5 2025, 13:08:55) [MSC v.1929 64 bit (AMD64)]
machine: AMD64 platform: Windows-10-10.0.26200-SP0
onnxruntime: 1.23.2
opencv: 4.13.0
scipy spline available: True


In [2]:
# ----- Path configuration -----
PROJECT_ROOT_LOCAL = Path(r"~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild")
SHARED_LANE_ROOT = Path(r"~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\20_shared_assets\dataset\lane")

REVIEW_ROOT_LOCAL = PROJECT_ROOT_LOCAL / "review_outputs" / "10_pi_runtime_latency_sequence_validation_v1"
DEFAULT_PACKAGE_ROOT_LOCAL = REVIEW_ROOT_LOCAL / "pkg"
NOTEBOOK_SRC_PATH = PROJECT_ROOT_LOCAL / "notebooks" / "10_pi_runtime_latency_sequence_validation_v1.ipynb"

# On Pi, open this notebook from inside the copied pkg folder.
PACKAGE_ROOT_OVERRIDE = None
cwd = Path.cwd()
if PACKAGE_ROOT_OVERRIDE:
    PACKAGE_ROOT = Path(PACKAGE_ROOT_OVERRIDE)
elif (cwd / "m" / "model_path.json").exists() and (cwd / "c").exists() and (cwd / "r").exists():
    PACKAGE_ROOT = cwd
else:
    PACKAGE_ROOT = DEFAULT_PACKAGE_ROOT_LOCAL

IS_PI = platform.machine().lower() in {"aarch64", "armv7l", "armv6l"} or "raspberry" in platform.platform().lower()
IS_LOCAL_PROJECT = PROJECT_ROOT_LOCAL.exists()
RUN_PREPARE_PACKAGE = bool(IS_LOCAL_PROJECT and not IS_PI)
RUN_VERIFY_PACKAGE = True
WRITE_SEQUENCE_VIDEO = True

# Geometry / contracts from 05-09.
RAW_W = 1296
RAW_H = 972
CUT_HEIGHT = 445
IMG_W = 800
IMG_H = 320
NUM_POINTS = 72
N_STRIPS = NUM_POINTS - 1
N_OFFSETS = NUM_POINTS
NUM_PRIORS = 192
OUTPUT_DIM = 78
SAMPLE_Y = list(range(971, 444, -20))
IMAGE_CENTER_X = RAW_W / 2.0

# Sample sizes for local/Pi parity package.
PARITY_VAL_FRAMES = 12
PARITY_FIELD3_FRAMES = 80
PARITY_HOLDOUT_FRAMES = 24
SEQUENCE_FIELD3_FRAMES = 240
PYTORCH_BASELINE_FRAMES = 40

# Runtime / tolerance settings.
ORT_WARMUP_RUNS = 5
DEFAULT_PI_ORT_THREADS = 4
RAW_MAX_ABS_TOL_PI = 0.02
RAW_MEAN_ABS_TOL_PI = 0.001
LANE_MAX_PAIR_DIST_TOL_PX = 2.0
STEER_MAX_ABS_TOL = 0.002
DEFAULT_PAIR_BONUS_PX = 60.0

print("IS_PI:", IS_PI)
print("RUN_PREPARE_PACKAGE:", RUN_PREPARE_PACKAGE)
print("PACKAGE_ROOT:", PACKAGE_ROOT)

IS_PI: False
RUN_PREPARE_PACKAGE: True
PACKAGE_ROOT: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild\review_outputs\10_pi_runtime_latency_sequence_validation_v1\pkg


In [3]:
# ----- Robust filesystem helpers -----
def fs_path(path):
    p = Path(path)
    s = str(p.resolve())
    if sys.platform.startswith("win") and not s.startswith("\\\\?\\"):
        return "\\\\?\\" + s
    return s

def exists_fs(path):
    return os.path.exists(fs_path(path))

def read_json(path):
    with open(fs_path(path), "r", encoding="utf-8") as f:
        return json.load(f)

def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(fs_path(path), "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def write_csv(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(fs_path(path), "w", encoding="utf-8-sig", newline="") as f:
        df.to_csv(f, index=False)

def copy_file(src, dst):
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    return shutil.copy2(fs_path(src), fs_path(dst))

def imread_bgr(path):
    data = np.fromfile(fs_path(path), dtype=np.uint8)
    img = cv2.imdecode(data, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Could not read image: {path}")
    return img

def imwrite_bgr(path, image):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    ok, buf = cv2.imencode(path.suffix or ".jpg", image)
    if not ok:
        raise RuntimeError(f"Could not encode image: {path}")
    buf.tofile(fs_path(path))
    return path

def safe_key(prefix, source_path, order):
    h = hashlib.sha1(str(source_path).replace("\\", "/").encode("utf-8")).hexdigest()[:10]
    return f"{prefix}_{int(order):04d}_{h}"

def even_sample(items, n):
    items = list(items)
    if n is None or len(items) <= n:
        return items
    idx = np.linspace(0, len(items) - 1, n).round().astype(int)
    out, seen = [], set()
    for i in idx:
        i = int(i)
        if i not in seen:
            out.append(items[i])
            seen.add(i)
    return out

def percentile_summary(values):
    arr = np.asarray(list(values), dtype=np.float64)
    if len(arr) == 0:
        return {"mean": None, "p50": None, "p90": None, "p95": None, "max": None}
    return {
        "mean": float(arr.mean()),
        "p50": float(np.percentile(arr, 50)),
        "p90": float(np.percentile(arr, 90)),
        "p95": float(np.percentile(arr, 95)),
        "max": float(arr.max()),
    }

## Runtime Core

아래 셀은 Pi에서도 그대로 쓰는 핵심 runtime이다.

- ONNX session 생성
- 학습과 동일한 preprocess
- 07 `official_overlap_python` decoder
- 08 `inside_soft` steering

이 셀 이후의 local/Pi 검증은 모두 이 함수들을 사용한다.

In [4]:
# ----- Contracts and ONNX Runtime -----
def load_package_contracts(package_root):
    package_root = Path(package_root)
    decode_contract = read_json(package_root / "c" / "decode_contract_v1.json")
    driving_contract = read_json(package_root / "c" / "driving_contract_v1.json")
    driving_contract["driving_contract"]["params"].setdefault("pair_bonus_px", DEFAULT_PAIR_BONUS_PX)
    onnx_report = read_json(package_root / "c" / "onnx_parity_report_v1.json")
    model_info = read_json(package_root / "m" / "model_path.json")
    return decode_contract, driving_contract, onnx_report, model_info

def make_ort_session(package_root, intra_op_num_threads=None):
    package_root = Path(package_root)
    _, _, _, model_info = load_package_contracts(package_root)
    model_path = package_root / model_info["onnx_rel"]
    data_path = package_root / model_info["external_data_rel"]
    assert exists_fs(model_path), model_path
    assert exists_fs(data_path), data_path

    if intra_op_num_threads is None and IS_PI:
        intra_op_num_threads = DEFAULT_PI_ORT_THREADS
    so = ort.SessionOptions()
    if intra_op_num_threads is not None:
        so.intra_op_num_threads = int(intra_op_num_threads)
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    session = ort.InferenceSession(fs_path(model_path), sess_options=so, providers=["CPUExecutionProvider"])
    return session, session.get_inputs()[0].name, session.get_outputs()[0].name

def preprocess_bgr_for_model(bgr):
    crop = bgr[int(CUT_HEIGHT):, :, :]
    resized = cv2.resize(crop, (IMG_W, IMG_H), interpolation=cv2.INTER_LINEAR)
    tensor = resized.astype(np.float32).transpose(2, 0, 1)[None, ...] / 255.0
    assert tensor.shape == (1, 3, IMG_H, IMG_W)
    return tensor

def run_onnx_raw(session, input_name, output_name, bgr):
    inp = preprocess_bgr_for_model(bgr)
    out = session.run([output_name], {input_name: inp})[0]
    assert out.shape == (1, NUM_PRIORS, OUTPUT_DIM), out.shape
    return out[0].astype(np.float32)

def warmup_ort_session(session, input_name, output_name, package_root, records, label="ORT"):
    if ORT_WARMUP_RUNS <= 0 or not records:
        return
    first = records[0]
    bgr = imread_bgr(Path(package_root) / first["image_rel"])
    inp = preprocess_bgr_for_model(bgr)
    for _ in range(int(ORT_WARMUP_RUNS)):
        _ = session.run([output_name], {input_name: inp})[0]
    print(f"{label} warmup runs:", ORT_WARMUP_RUNS)

# ----- 07 standalone decoder -----
def softmax_positive_score(logits_2):
    logits = logits_2.astype(np.float32)
    logits = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(logits)
    return exp[:, 1] / exp.sum(axis=1)

def official_cuda_suppresses(a, b, threshold):
    start_a = int(float(a[2]) * N_STRIPS + 0.5)
    start_b = int(float(b[2]) * N_STRIPS + 0.5)
    start = max(start_a, start_b)
    len_a, len_b = float(a[4]), float(b[4])
    end_a = int(start_a + len_a - 1 + 0.5 - (1 if (len_a - 1) < 0 else 0))
    end_b = int(start_b + len_b - 1 + 0.5 - (1 if (len_b - 1) < 0 else 0))
    end = min(end_a, end_b, N_OFFSETS - 1)
    if end < start:
        return False
    dist = float(np.abs(a[5 + start:5 + end + 1] - b[5 + start:5 + end + 1]).sum())
    return dist < float(threshold) * float(end - start + 1)

def official_overlap_nms(nms_predictions, scores, nms_thres, top_k):
    order = np.argsort(-scores)
    kept = []
    for idx in order:
        duplicate = any(official_cuda_suppresses(nms_predictions[idx], nms_predictions[j], nms_thres) for j in kept)
        if not duplicate:
            kept.append(int(idx))
        if len(kept) >= int(top_k):
            break
    return kept

def resample_lane_xs(xs, ys, query_ys):
    if len(xs) < 2:
        return np.full_like(query_ys, -2.0, dtype=np.float32)
    order = np.argsort(ys)
    ys_sorted, xs_sorted = ys[order], xs[order]
    spline = InterpolatedUnivariateSpline(ys_sorted, xs_sorted, k=min(3, len(xs_sorted) - 1))
    out = spline(query_ys).astype(np.float32)
    min_y, max_y = float(ys_sorted.min()) - 0.01, float(ys_sorted.max()) + 0.01
    out[(query_ys < min_y) | (query_ys > max_y)] = -2.0
    return out

def prediction_to_lane_array(prediction, confidence):
    pred = prediction.copy().astype(np.float32)
    lane_xs = pred[6:].copy()
    start = min(max(0, int(round(float(pred[2]) * N_STRIPS))), N_STRIPS)
    length = int(round(float(pred[5])))
    end = min(start + length - 1, N_OFFSETS - 1)
    lane_xs[end + 1:] = -2.0
    if start > 0:
        valid_prefix = ((lane_xs[:start] >= 0.0) & (lane_xs[:start] <= 1.0)).astype(np.int32)
        mask = ~(valid_prefix[::-1].cumprod()[::-1].astype(bool))
        prefix = lane_xs[:start].copy()
        prefix[mask] = -2.0
        lane_xs[:start] = prefix

    prior_ys = np.linspace(1.0, 0.0, N_OFFSETS, dtype=np.float32)
    valid = lane_xs >= 0.0
    if int(valid.sum()) <= 1:
        return None
    xs_norm = lane_xs[valid][::-1].astype(np.float32)
    ys_crop_norm = prior_ys[valid][::-1].astype(np.float32)
    ys_norm = (ys_crop_norm * float(RAW_H - CUT_HEIGHT) + float(CUT_HEIGHT)) / float(RAW_H)

    sample_ys_norm = np.array(SAMPLE_Y, dtype=np.float32) / float(RAW_H)
    sample_xs_norm = resample_lane_xs(xs_norm, ys_norm, sample_ys_norm)
    valid_sample = (sample_xs_norm >= 0.0) & (sample_xs_norm < 1.0)
    if int(valid_sample.sum()) <= 1:
        return None
    pts = np.stack([sample_xs_norm[valid_sample] * float(RAW_W), sample_ys_norm[valid_sample] * float(RAW_H)], axis=1)
    return {"points": pts.astype(np.float32), "conf": float(confidence)}

def decode_raw_to_lanes(raw_predictions, decode_contract):
    dc = decode_contract.get("decoder_contract", decode_contract)
    conf_threshold = float(dc["conf_threshold"])
    nms_thres = float(dc["nms_thres"])
    nms_topk = int(dc["nms_topk"])
    predictions = raw_predictions.astype(np.float32)
    scores = softmax_positive_score(predictions[:, :2])
    keep_mask = scores >= conf_threshold
    if int(keep_mask.sum()) == 0:
        return []
    pred_kept, score_kept = predictions[keep_mask].copy(), scores[keep_mask].copy()
    nms_predictions = np.concatenate([pred_kept[:, :4], pred_kept[:, 5:]], axis=1).astype(np.float32)
    nms_predictions[:, 4] *= float(N_STRIPS)
    nms_predictions[:, 5:] *= float(IMG_W - 1)
    keep = official_overlap_nms(nms_predictions, score_kept, nms_thres, nms_topk)
    selected, selected_scores = pred_kept[keep].copy(), score_kept[keep].copy()
    selected[:, 5] = np.round(selected[:, 5] * float(N_STRIPS))
    lanes = []
    for pred, score in zip(selected, selected_scores):
        lane = prediction_to_lane_array(pred, score)
        if lane is not None:
            lanes.append(lane)
    return lanes

def lanes_to_jsonable(lanes):
    return [{"conf": float(l.get("conf", 0.0)), "points": np.asarray(l["points"], dtype=float).round(4).tolist()} for l in lanes]

def lanes_from_jsonable(items):
    return [{"conf": float(x.get("conf", 0.0)), "points": np.asarray(x["points"], dtype=np.float32)} for x in items]

# ----- 08 inside_soft steering -----
def interp_or_nearest_x(points, y, max_y_distance):
    pts = np.asarray(points, dtype=np.float32)
    if len(pts) == 0:
        return None
    xs, ys = pts[:, 0], pts[:, 1]
    order = np.argsort(ys)
    ys, xs = ys[order], xs[order]
    if float(ys.min()) <= y <= float(ys.max()) and len(pts) >= 2:
        return float(np.interp(y, ys, xs))
    nearest_idx = int(np.argmin(np.abs(ys - y)))
    if abs(float(ys[nearest_idx]) - float(y)) <= float(max_y_distance):
        return float(xs[nearest_idx])
    return None

def feature_from_lane(lane, pp):
    pts = np.asarray(lane["points"], dtype=np.float32)
    if len(pts) < int(pp["min_points"]):
        return None
    if float(pts[:, 1].max() - pts[:, 1].min()) < float(pp["min_y_span"]):
        return None
    y_top = float((RAW_H - 1) * pp["top_ratio"])
    y_primary = float((RAW_H - 1) * pp["primary_ratio"])
    y_bottom = float((RAW_H - 1) * pp["bottom_ratio"])
    x_top = interp_or_nearest_x(pts, y_top, pp["max_y_distance"])
    x_primary = interp_or_nearest_x(pts, y_primary, pp["max_y_distance"])
    x_bottom = interp_or_nearest_x(pts, y_bottom, pp["max_y_distance"])
    if x_top is None or x_primary is None or x_bottom is None:
        return None
    return {"x_top": x_top, "x_primary": x_primary, "x_bottom": x_bottom, "heading": (x_top - x_bottom) / max(1.0, y_bottom - y_top), "y_top": y_top, "y_primary": y_primary, "y_bottom": y_bottom}

def estimate_steer(center_x, heading, pp):
    pos_error = (float(center_x) - IMAGE_CENTER_X) / IMAGE_CENTER_X
    raw = float(pp["steer_gain"]) * (float(pp["k_pos"]) * pos_error + float(pp["k_heading"]) * float(heading))
    return float(np.clip(raw, -float(pp["max_steer_norm"]), float(pp["max_steer_norm"])))

def trajectory_from_pair(left, right):
    center_top = 0.5 * (left["x_top"] + right["x_top"])
    center_primary = 0.5 * (left["x_primary"] + right["x_primary"])
    center_bottom = 0.5 * (left["x_bottom"] + right["x_bottom"])
    return {"center_primary": center_primary, "heading": (center_top - center_bottom) / max(1.0, left["y_bottom"] - left["y_top"]), "mode": "both_stable"}

def trajectory_from_single(feat, side, pp):
    sign = 1.0 if side == "left" else -1.0
    offset = float(pp["safe_offset_px"])
    center_top = feat["x_top"] + sign * offset
    center_primary = feat["x_primary"] + sign * offset
    center_bottom = feat["x_bottom"] + sign * offset
    return {"center_primary": center_primary, "heading": (center_top - center_bottom) / max(1.0, feat["y_bottom"] - feat["y_top"]), "mode": f"single_{side}", "source_x": feat["x_primary"]}

def init_drive_memory():
    return {"smoothed_center_x": IMAGE_CENTER_X, "smoothed_heading": 0.0, "last_steer_norm": 0.0, "turn_bias": 0.0, "lost_frames": 0, "unstable_frames": 0}

def choose_trajectory(features, memory, pp):
    if not features:
        return None
    lefts = [f for f in features if f["x_primary"] < IMAGE_CENTER_X]
    rights = [f for f in features if f["x_primary"] >= IMAGE_CENTER_X]
    left = max(lefts, key=lambda f: f["x_primary"]) if lefts else None
    right = min(rights, key=lambda f: f["x_primary"]) if rights else None
    candidates = []
    if left is not None and right is not None:
        gap = right["x_primary"] - left["x_primary"]
        if float(pp["gap_min_px"]) <= gap <= float(pp["gap_max_px"]):
            candidates.append(trajectory_from_pair(left, right))
    for feat in features:
        for side in ["left", "right"]:
            cand = trajectory_from_single(feat, side, pp)
            penalty = 0.0
            if feat["x_primary"] < IMAGE_CENTER_X - float(pp["single_side_deadband_px"]) and side == "right":
                penalty = float(pp["side_prior_penalty_px"])
            if feat["x_primary"] > IMAGE_CENTER_X + float(pp["single_side_deadband_px"]) and side == "left":
                penalty = float(pp["side_prior_penalty_px"])
            cand["side_penalty"] = penalty
            candidates.append(cand)
    prev_center = memory.get("smoothed_center_x", IMAGE_CENTER_X)
    prev_heading = memory.get("smoothed_heading", 0.0)
    prev_steer = memory.get("last_steer_norm", 0.0)
    def cost(c):
        center_cost = abs(c["center_primary"] - prev_center)
        heading_cost = abs(c["heading"] - prev_heading) * float(pp["heading_cost_px"])
        steer_cost = abs(estimate_steer(c["center_primary"], c["heading"], pp) - prev_steer) * float(pp["steer_cost_px"])
        pair_bonus = -float(pp.get("pair_bonus_px", DEFAULT_PAIR_BONUS_PX)) if c["mode"] == "both_stable" else 0.0
        return center_cost + heading_cost + steer_cost + c.get("side_penalty", 0.0) + pair_bonus
    best = min(candidates, key=cost)
    best["candidate_count"] = len(candidates)
    best["candidate_cost"] = float(cost(best))
    return best

def update_drive(lanes, memory, driving_contract):
    pp = driving_contract["driving_contract"]["params"] if "driving_contract" in driving_contract else driving_contract["params"]
    features = [f for f in (feature_from_lane(l, pp) for l in lanes) if f is not None]
    measured = choose_trajectory(features, memory, pp)
    raw_mode = "lost" if measured is None else measured["mode"]
    if measured is None:
        memory["lost_frames"] += 1
        memory["unstable_frames"] = 0
        if memory["lost_frames"] <= int(pp["lost_short_frames"]):
            mode = "lost_short_recovery"
            target = memory["last_steer_norm"] * float(pp["recovery_decay"])
        else:
            mode = "lost_active_recovery"
            sign = float(np.sign(memory["turn_bias"])) or float(np.sign(memory["last_steer_norm"]))
            ramp = float(pp["recovery_ramp"]) * max(0, memory["lost_frames"] - int(pp["lost_short_frames"]))
            target = memory["last_steer_norm"] * float(pp["recovery_decay"]) + float(pp["recovery_gain"]) * memory["turn_bias"] + ramp * sign
        target = float(np.clip(target, -float(pp["max_steer_norm"]), float(pp["max_steer_norm"])))
        memory["smoothed_heading"] *= float(pp["lost_heading_decay"])
        steer = float(pp["steer_alpha"]) * target + (1.0 - float(pp["steer_alpha"])) * memory["last_steer_norm"]
        steer = float(np.clip(steer, -float(pp["max_steer_norm"]), float(pp["max_steer_norm"])))
        memory["last_steer_norm"] = steer
        memory["turn_bias"] = float(pp["turn_bias_alpha"]) * steer + (1.0 - float(pp["turn_bias_alpha"])) * memory["turn_bias"]
        return {"raw_mode": raw_mode, "effective_mode": mode, "feature_count": len(features), "candidate_count": 0, "measured_center_x": np.nan, "measured_heading": np.nan, "smoothed_center_x": memory["smoothed_center_x"], "smoothed_heading": memory["smoothed_heading"], "steer_norm": steer, "turn_bias": memory["turn_bias"], "lost_frames": memory["lost_frames"]}

    memory["lost_frames"] = 0
    center, heading = float(measured["center_primary"]), float(measured["heading"])
    center_jump = abs(center - memory["smoothed_center_x"])
    heading_jump = abs(heading - memory["smoothed_heading"])
    unstable = center_jump > float(pp["jump_center_px"]) or heading_jump > float(pp["jump_heading"])
    if unstable:
        memory["unstable_frames"] += 1
        blend = min(float(pp["jump_blend_max"]), float(pp["jump_blend_start"]) + float(pp["jump_blend_step"]) * max(0, memory["unstable_frames"] - 1))
        effective_center = (1.0 - blend) * memory["smoothed_center_x"] + blend * center
        effective_heading = (1.0 - blend) * memory["smoothed_heading"] + blend * heading
        mode = "unstable_blend"
    else:
        memory["unstable_frames"] = 0
        effective_center, effective_heading, mode = center, heading, measured["mode"]
    memory["smoothed_center_x"] = float(pp["center_alpha"]) * effective_center + (1.0 - float(pp["center_alpha"])) * memory["smoothed_center_x"]
    memory["smoothed_heading"] = float(pp["heading_alpha"]) * effective_heading + (1.0 - float(pp["heading_alpha"])) * memory["smoothed_heading"]
    target = estimate_steer(memory["smoothed_center_x"], memory["smoothed_heading"], pp)
    steer = float(pp["steer_alpha"]) * target + (1.0 - float(pp["steer_alpha"])) * memory["last_steer_norm"]
    steer = float(np.clip(steer, -float(pp["max_steer_norm"]), float(pp["max_steer_norm"])))
    memory["last_steer_norm"] = steer
    memory["turn_bias"] = float(pp["turn_bias_alpha"]) * steer + (1.0 - float(pp["turn_bias_alpha"])) * memory["turn_bias"]
    return {"raw_mode": raw_mode, "effective_mode": mode, "feature_count": len(features), "candidate_count": int(measured.get("candidate_count", 0)), "measured_center_x": center, "measured_heading": heading, "smoothed_center_x": memory["smoothed_center_x"], "smoothed_heading": memory["smoothed_heading"], "steer_norm": steer, "turn_bias": memory["turn_bias"], "lost_frames": memory["lost_frames"]}

## Local Package Preparation

Local 실행 시 `pkg/` 패키지를 만든다. 이 폴더 전체를 Pi로 복사하면 된다.

In [5]:
def image_paths_under(root):
    exts = {".jpg", ".jpeg", ".png", ".bmp"}
    root = Path(root)
    return sorted([p for p in root.rglob("*") if p.suffix.lower() in exts], key=lambda p: str(p)) if root.exists() else []

def collect_local_records():
    dataset_root = SHARED_LANE_ROOT / "map_culane_localfit_train_field1_field2_v1"
    raw_field3_root = SHARED_LANE_ROOT / "raw" / "field3"
    holdout_root = SHARED_LANE_ROOT / "holdout"
    val_lines = [line.strip() for line in (dataset_root / "list" / "val.txt").read_text(encoding="utf-8").splitlines() if line.strip()]
    val_paths = [dataset_root / line.lstrip("/").replace("/", os.sep) for line in val_lines]
    field3_paths = image_paths_under(raw_field3_root)
    holdout_paths = image_paths_under(holdout_root)
    records = []
    def add(set_name, role, paths, limit, prefix):
        for order, src in enumerate(even_sample(paths, limit)):
            key = safe_key(prefix, src, order)
            records.append({"key": key, "set": set_name, "role": role, "order": order, "image_rel": f"i/{key}{Path(src).suffix.lower()}", "source_path_local": str(src), "source_name": Path(src).name})
    add("val", "parity", val_paths, PARITY_VAL_FRAMES, "val")
    add("field3", "parity", field3_paths, PARITY_FIELD3_FRAMES, "f3p")
    add("holdout", "parity", holdout_paths, PARITY_HOLDOUT_FRAMES, "hold")
    add("field3", "sequence", field3_paths, SEQUENCE_FIELD3_FRAMES, "f3s")
    return records

def prepare_package():
    if not RUN_PREPARE_PACKAGE:
        print("Skipping local package preparation.")
        return
    out = PACKAGE_ROOT
    for sub in ["m", "c", "i", "r", "t", "out_local", "out_pi", "v"]:
        (out / sub).mkdir(parents=True, exist_ok=True)
    print("package root:", out)

    models_src = PROJECT_ROOT_LOCAL / "review_outputs" / "09_onnx_export_parity_v1" / "models"
    onnx_files = sorted(models_src.glob("*.onnx"))
    assert onnx_files, models_src
    onnx_src = onnx_files[0]
    data_src = onnx_src.with_name(onnx_src.name + ".data")
    assert exists_fs(data_src), data_src
    onnx_dst = out / "m" / onnx_src.name
    data_dst = out / "m" / data_src.name
    copy_file(onnx_src, onnx_dst)
    copy_file(data_src, data_dst)
    write_json(out / "m" / "model_path.json", {"onnx_rel": f"m/{onnx_src.name}", "external_data_rel": f"m/{data_src.name}", "note": "Keep ONNX and external .onnx.data together."})

    contract_files = {
        "decode_contract_v1.json": PROJECT_ROOT_LOCAL / "review_outputs" / "07_decode_contract_v1" / "decode_contract_v1.json",
        "driving_contract_v1.json": PROJECT_ROOT_LOCAL / "review_outputs" / "08_driving_postprocess_contract_v1" / "inside_memory_driving_postprocess_contract_candidate.json",
        "onnx_parity_report_v1.json": PROJECT_ROOT_LOCAL / "review_outputs" / "09_onnx_export_parity_v1" / "onnx_parity_report_v1.json",
    }
    for name, src in contract_files.items():
        assert exists_fs(src), src
        copy_file(src, out / "c" / name)
    if exists_fs(NOTEBOOK_SRC_PATH):
        copy_file(NOTEBOOK_SRC_PATH, out / NOTEBOOK_SRC_PATH.name)

    records = collect_local_records()
    for rec in records:
        copy_file(rec["source_path_local"], out / rec["image_rel"])
    manifest = pd.DataFrame(records)
    write_csv(manifest, out / "t" / "records_manifest.csv")
    print(manifest.groupby(["set", "role"]).size())

prepare_package()

package root: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild\review_outputs\10_pi_runtime_latency_sequence_validation_v1\pkg
set      role    
field3   parity       80
         sequence    240
holdout  parity       24
val      parity       12
dtype: int64


## Local ONNX References

Reference는 PyTorch가 아니라 local ONNX Runtime으로 만든다. 따라서 Pi 검증은 `local ONNX ↔ Pi ONNX` 비교가 된다.

In [6]:
def load_records_manifest(package_root):
    path = Path(package_root) / "t" / "records_manifest.csv"
    assert exists_fs(path), path
    return pd.read_csv(fs_path(path)).to_dict("records")

def generate_local_references():
    if not RUN_PREPARE_PACKAGE:
        print("Skipping local reference generation.")
        return
    decode_contract, driving_contract, onnx_report, model_info = load_package_contracts(PACKAGE_ROOT)
    records = load_records_manifest(PACKAGE_ROOT)
    session, input_name, output_name = make_ort_session(PACKAGE_ROOT)
    warmup_ort_session(session, input_name, output_name, PACKAGE_ROOT, records, label="reference ORT")

    raw_refs, decoded_rows, raw_summary = {}, [], []
    for i, rec in enumerate(tqdm(records, desc="local reference")):
        bgr = imread_bgr(PACKAGE_ROOT / rec["image_rel"])
        raw = run_onnx_raw(session, input_name, output_name, bgr)
        lanes = decode_raw_to_lanes(raw, decode_contract)
        raw_refs[rec["key"]] = raw
        decoded_rows.append({"key": rec["key"], "set": rec["set"], "role": rec["role"], "order": int(rec["order"]), "lanes": lanes_to_jsonable(lanes)})
        raw_summary.append({"key": rec["key"], "set": rec["set"], "role": rec["role"], "lane_count": len(lanes), "raw_min": float(raw.min()), "raw_max": float(raw.max()), "raw_mean": float(raw.mean())})

    np.savez_compressed(fs_path(PACKAGE_ROOT / "r" / "ref_raw.npz"), **raw_refs)
    write_csv(pd.DataFrame(raw_summary), PACKAGE_ROOT / "r" / "ref_raw_summary.csv")
    with open(fs_path(PACKAGE_ROOT / "r" / "ref_decoded.jsonl"), "w", encoding="utf-8") as f:
        for row in decoded_rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    lanes_by_key = {r["key"]: lanes_from_jsonable(r["lanes"]) for r in decoded_rows}
    memory = init_drive_memory()
    steer_rows = []
    seq_records = sorted([r for r in records if r["role"] == "sequence"], key=lambda r: int(r["order"]))
    for rec in seq_records:
        row = update_drive(lanes_by_key[rec["key"]], memory, driving_contract)
        row.update({"key": rec["key"], "set": rec["set"], "role": rec["role"], "order": int(rec["order"])})
        steer_rows.append(row)
    write_csv(pd.DataFrame(steer_rows), PACKAGE_ROOT / "r" / "ref_steer_seq.csv")

    manifest = {"created_at": time.strftime("%Y-%m-%d %H:%M:%S"), "source": "local ONNX reference", "records": len(records), "sequence_records": len(seq_records), "decoder_resampler": "official_scipy_spline", "preprocess": onnx_report["preprocess_contract"], "decoder": decode_contract["decoder_contract"], "driving": driving_contract["driving_contract"]}
    write_json(PACKAGE_ROOT / "r" / "ref_manifest.json", manifest)
    print("references written:", PACKAGE_ROOT / "r")

generate_local_references()

reference ORT warmup runs: 5


local reference:   0%|          | 0/356 [00:00<?, ?it/s]

references written: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild\review_outputs\10_pi_runtime_latency_sequence_validation_v1\pkg\r


## Runtime Verification

Local에서는 sanity check, Pi에서는 실제 검증이다.

In [7]:
def load_reference_decoded(package_root):
    rows = []
    with open(fs_path(Path(package_root) / "r" / "ref_decoded.jsonl"), "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                row = json.loads(line)
                rows.append(row)
    return {row["key"]: lanes_from_jsonable(row["lanes"]) for row in rows}

def lane_pair_distance(lanes_a, lanes_b):
    if len(lanes_a) != len(lanes_b):
        return math.inf
    if not lanes_a:
        return 0.0
    vals = []
    for a, b in zip(lanes_a, lanes_b):
        pa, pb = np.asarray(a["points"], dtype=np.float32), np.asarray(b["points"], dtype=np.float32)
        n = min(len(pa), len(pb))
        vals.append(math.inf if n == 0 else float(np.linalg.norm(pa[:n] - pb[:n], axis=1).mean()))
    return float(max(vals)) if vals else 0.0

def verify_package():
    if not RUN_VERIFY_PACKAGE:
        return None
    assert exists_fs(PACKAGE_ROOT / "r" / "ref_raw.npz"), "Local references are missing. Run local preparation first."
    decode_contract, driving_contract, onnx_report, model_info = load_package_contracts(PACKAGE_ROOT)
    records = load_records_manifest(PACKAGE_ROOT)
    session, input_name, output_name = make_ort_session(PACKAGE_ROOT)
    warmup_ort_session(session, input_name, output_name, PACKAGE_ROOT, records, label="verify ORT")

    ref_raw = np.load(fs_path(PACKAGE_ROOT / "r" / "ref_raw.npz"))
    ref_lanes = load_reference_decoded(PACKAGE_ROOT)
    ref_steer = pd.read_csv(fs_path(PACKAGE_ROOT / "r" / "ref_steer_seq.csv"))
    ref_steer_by_key = {r["key"]: r for r in ref_steer.to_dict("records")}

    raw_rows, decode_rows, latency_rows, steer_rows = [], [], [], []
    seq_lanes_by_key = {}
    memory = init_drive_memory()
    for rec in tqdm(records, desc="runtime verify"):
        key = rec["key"]
        t0 = time.perf_counter()
        bgr = imread_bgr(PACKAGE_ROOT / rec["image_rel"])
        t1 = time.perf_counter()
        inp = preprocess_bgr_for_model(bgr)
        t2 = time.perf_counter()
        raw = session.run([output_name], {input_name: inp})[0][0].astype(np.float32)
        t3 = time.perf_counter()
        lanes = decode_raw_to_lanes(raw, decode_contract)
        t4 = time.perf_counter()
        steer_ms = 0.0
        latency = {"key": key, "set": rec["set"], "role": rec["role"], "order": int(rec["order"]), "read_ms": (t1 - t0) * 1000, "preprocess_ms": (t2 - t1) * 1000, "inference_ms": (t3 - t2) * 1000, "decode_ms": (t4 - t3) * 1000}
        if rec["role"] == "parity":
            d = np.abs(raw - ref_raw[key].astype(np.float32))
            raw_rows.append({"key": key, "set": rec["set"], "max_abs_diff": float(d.max()), "mean_abs_diff": float(d.mean()), "p99_abs_diff": float(np.percentile(d, 99))})
            decode_rows.append({"key": key, "set": rec["set"], "count_local": len(ref_lanes[key]), "count_runtime": len(lanes), "count_mismatch": int(len(ref_lanes[key]) != len(lanes)), "max_pair_dist_px": lane_pair_distance(ref_lanes[key], lanes)})
        if rec["role"] == "sequence":
            seq_lanes_by_key[key] = lanes
            ts0 = time.perf_counter()
            row = update_drive(lanes, memory, driving_contract)
            ts1 = time.perf_counter()
            steer_ms = (ts1 - ts0) * 1000
            row.update({"key": key, "set": rec["set"], "role": rec["role"], "order": int(rec["order"])})
            steer_rows.append(row)
        latency["steering_ms"] = steer_ms
        latency_rows.append(latency)

    out_dir = PACKAGE_ROOT / ("out_pi" if IS_PI else "out_local")
    out_dir.mkdir(parents=True, exist_ok=True)
    raw_df, decode_df, latency_df, steer_df = pd.DataFrame(raw_rows), pd.DataFrame(decode_rows), pd.DataFrame(latency_rows), pd.DataFrame(steer_rows)
    write_csv(raw_df, out_dir / "raw_parity.csv")
    write_csv(decode_df, out_dir / "decode_parity.csv")
    write_csv(latency_df, out_dir / "latency_breakdown.csv")
    write_csv(steer_df, out_dir / "steering_sequence.csv")

    steer_cmp = []
    for row in steer_rows:
        ref = ref_steer_by_key[row["key"]]
        steer_cmp.append({"key": row["key"], "order": int(row["order"]), "mode_local": ref["effective_mode"], "mode_runtime": row["effective_mode"], "mode_mismatch": int(ref["effective_mode"] != row["effective_mode"]), "steer_abs_diff": abs(float(ref["steer_norm"]) - float(row["steer_norm"])), "center_abs_diff": abs(float(ref["smoothed_center_x"]) - float(row["smoothed_center_x"])), "heading_abs_diff": abs(float(ref["smoothed_heading"]) - float(row["smoothed_heading"]))})
    steer_cmp_df = pd.DataFrame(steer_cmp)
    write_csv(steer_cmp_df, out_dir / "steering_parity.csv")

    latency_df["pipeline_ms"] = latency_df["preprocess_ms"] + latency_df["inference_ms"] + latency_df["decode_ms"] + latency_df["steering_ms"]
    latency_df["offline_total_ms"] = latency_df["read_ms"] + latency_df["pipeline_ms"]
    summary = {"created_at": time.strftime("%Y-%m-%d %H:%M:%S"), "environment": {"is_pi": IS_PI, "machine": platform.machine(), "platform": platform.platform(), "python": sys.version, "onnxruntime": ort.__version__, "opencv": cv2.__version__, "scipy": HAS_SCIPY, "ort_warmup_runs": ORT_WARMUP_RUNS, "pi_ort_threads": DEFAULT_PI_ORT_THREADS}, "records": {"total": len(records), "parity": int((pd.DataFrame(records)["role"] == "parity").sum()), "sequence": int((pd.DataFrame(records)["role"] == "sequence").sum())}, "raw_parity": {"max_abs_diff": float(raw_df["max_abs_diff"].max()), "mean_abs_diff_max": float(raw_df["mean_abs_diff"].max()), "pass": bool(raw_df["max_abs_diff"].max() <= RAW_MAX_ABS_TOL_PI and raw_df["mean_abs_diff"].max() <= RAW_MEAN_ABS_TOL_PI), "tolerance": {"max_abs": RAW_MAX_ABS_TOL_PI, "mean_abs": RAW_MEAN_ABS_TOL_PI}}, "decode_parity": {"count_mismatch": int(decode_df["count_mismatch"].sum()), "max_pair_dist_px": float(decode_df["max_pair_dist_px"].replace([np.inf, -np.inf], np.nan).max()), "pass": bool(decode_df["count_mismatch"].sum() == 0 and decode_df["max_pair_dist_px"].replace([np.inf, -np.inf], np.nan).max() <= LANE_MAX_PAIR_DIST_TOL_PX)}, "steering_parity": {"mode_mismatch": int(steer_cmp_df["mode_mismatch"].sum()), "max_steer_abs_diff": float(steer_cmp_df["steer_abs_diff"].max()), "pass": bool(steer_cmp_df["mode_mismatch"].sum() == 0 and steer_cmp_df["steer_abs_diff"].max() <= STEER_MAX_ABS_TOL)}, "latency_ms": {"read": percentile_summary(latency_df["read_ms"]), "preprocess": percentile_summary(latency_df["preprocess_ms"]), "inference": percentile_summary(latency_df["inference_ms"]), "decode": percentile_summary(latency_df["decode_ms"]), "steering": percentile_summary(latency_df["steering_ms"]), "pipeline_no_disk": percentile_summary(latency_df["pipeline_ms"]), "offline_total_with_disk": percentile_summary(latency_df["offline_total_ms"]), "fps_no_disk_mean": float(1000.0 / latency_df["pipeline_ms"].mean()), "fps_with_disk_mean": float(1000.0 / latency_df["offline_total_ms"].mean())}}
    summary["overall_pass"] = bool(summary["raw_parity"]["pass"] and summary["decode_parity"]["pass"] and summary["steering_parity"]["pass"])
    local_baseline_path = PACKAGE_ROOT / "r" / "local_onnx_latency_baseline.json"
    if IS_PI and exists_fs(local_baseline_path):
        local_baseline = read_json(local_baseline_path)
        local_mean = float(local_baseline["latency_ms"]["pipeline_no_disk"]["mean"])
        pi_mean = float(summary["latency_ms"]["pipeline_no_disk"]["mean"])
        summary["pi_vs_local_onnx_pipeline_mean_ratio"] = pi_mean / local_mean if local_mean > 0 else None
    if not IS_PI:
        write_json(local_baseline_path, {"created_at": summary["created_at"], "source": "local ONNX runtime baseline", "environment": summary["environment"], "latency_ms": summary["latency_ms"]})
    write_json(out_dir / "runtime_validation_report.json", summary)
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    return {"out_dir": out_dir, "summary": summary, "latency_df": latency_df, "steer_df": steer_df, "seq_lanes_by_key": seq_lanes_by_key, "records": records}

verify_result = verify_package()

verify ORT warmup runs: 5


runtime verify:   0%|          | 0/356 [00:00<?, ?it/s]

{
  "created_at": "2026-05-10 11:23:06",
  "environment": {
    "is_pi": false,
    "machine": "AMD64",
    "platform": "Windows-10-10.0.26200-SP0",
    "python": "3.10.18 | packaged by Anaconda, Inc. | (main, Jun  5 2025, 13:08:55) [MSC v.1929 64 bit (AMD64)]",
    "onnxruntime": "1.23.2",
    "opencv": "4.13.0",
    "scipy": true,
    "ort_warmup_runs": 5,
    "pi_ort_threads": 4
  },
  "records": {
    "total": 356,
    "parity": 116,
    "sequence": 240
  },
  "raw_parity": {
    "max_abs_diff": 0.0,
    "mean_abs_diff_max": 0.0,
    "pass": true,
    "tolerance": {
      "max_abs": 0.02,
      "mean_abs": 0.001
    }
  },
  "decode_parity": {
    "count_mismatch": 0,
    "max_pair_dist_px": 4.164377969573252e-05,
    "pass": true
  },
  "steering_parity": {
    "mode_mismatch": 0,
    "max_steer_abs_diff": 1.8361701756286486e-07,
    "pass": true
  },
  "latency_ms": {
    "read": {
      "mean": 7.33675421339653,
      "p50": 6.882999998197192,
      "p90": 9.366199999931268,
   

## Optional Local PyTorch `.pth` Latency Baseline

이 값은 참고용이다. Pi 배포 경로는 ONNX Runtime이므로, Pi와 직접 비교할 기준은 local ONNX latency다.

In [8]:
def benchmark_local_pytorch_pth_forward():
    if not RUN_PREPARE_PACKAGE:
        print("Skipping local PyTorch baseline outside local prepare mode.")
        return None
    try:
        import torch
        run_dir = PROJECT_ROOT_LOCAL / "colab_outputs" / "MapLane_LocalFit_Field12_v1_full" / "20260509_102232_lr_1e-04_b_8"
        code_dir = run_dir / "code"
        config_path = PROJECT_ROOT_LOCAL / "colab_outputs" / "meta" / "MapLane_LocalFit_Field12_v1_ResNet18_full.py"
        best_ckpt = run_dir / "ckpt" / "best.pth"
        if not (exists_fs(code_dir) and exists_fs(config_path) and exists_fs(best_ckpt)):
            raise FileNotFoundError("code/config/checkpoint missing")
        for module_name in list(sys.modules.keys()):
            if module_name == "clrkd" or module_name.startswith("clrkd."):
                del sys.modules[module_name]
        sys.path.insert(0, str(code_dir))
        from clrkd.utils.config import Config
        import clrkd.models
        from clrkd.models.registry import build_net
        text = Path(config_path).read_text(encoding="utf-8")
        namespace = {}
        exec(compile(text, str(config_path), "exec"), namespace)
        cfg_dict = {k: v for k, v in namespace.items() if not k.startswith("__")}
        cfg_local = Config(cfg_dict, cfg_text=text, filename=str(config_path))
        model = build_net(cfg_local).cpu().eval()
        ckpt = torch.load(fs_path(best_ckpt), map_location="cpu")
        state = ckpt["net"] if isinstance(ckpt, dict) and "net" in ckpt else ckpt
        state = {k.replace("module.", "", 1): v for k, v in state.items()}
        missing, unexpected = model.load_state_dict(state, strict=False)
        assert len(missing) == 0 and len(unexpected) == 0
        class Wrapper(torch.nn.Module):
            def __init__(self, detector):
                super().__init__()
                self.detector = detector
            def forward(self, img):
                return self.detector({"img": img})
        wrapper = Wrapper(model).eval()
        records = even_sample([r for r in load_records_manifest(PACKAGE_ROOT) if r["role"] == "parity"], PYTORCH_BASELINE_FRAMES)
        rows = []
        if records:
            inp0 = torch.from_numpy(preprocess_bgr_for_model(imread_bgr(PACKAGE_ROOT / records[0]["image_rel"])))
            with torch.no_grad():
                for _ in range(3):
                    _ = wrapper(inp0)
        for rec in tqdm(records, desc="local PyTorch latency"):
            bgr = imread_bgr(PACKAGE_ROOT / rec["image_rel"])
            t0 = time.perf_counter()
            inp = preprocess_bgr_for_model(bgr)
            t1 = time.perf_counter()
            with torch.no_grad():
                _ = wrapper(torch.from_numpy(inp)).detach().cpu().numpy()
            t2 = time.perf_counter()
            rows.append({"key": rec["key"], "set": rec["set"], "preprocess_ms": (t1 - t0) * 1000, "pytorch_forward_ms": (t2 - t1) * 1000, "pipeline_ms": (t2 - t0) * 1000})
        df = pd.DataFrame(rows)
        out = PACKAGE_ROOT / "out_local" / "local_pytorch_pth_forward_latency.csv"
        write_csv(df, out)
        summary = {"created_at": time.strftime("%Y-%m-%d %H:%M:%S"), "source": "local PyTorch best.pth CPU forward baseline", "frames": len(df), "latency_ms": {"preprocess": percentile_summary(df["preprocess_ms"]), "pytorch_forward": percentile_summary(df["pytorch_forward_ms"]), "pipeline": percentile_summary(df["pipeline_ms"]), "fps_pipeline_mean": float(1000.0 / df["pipeline_ms"].mean())}, "note": "Reference only; Pi deployment uses ONNX Runtime.", "csv": str(out)}
        write_json(PACKAGE_ROOT / "r" / "local_pytorch_pth_latency_baseline.json", summary)
        print(json.dumps(summary, ensure_ascii=False, indent=2))
        return summary
    except Exception as exc:
        skip = {"created_at": time.strftime("%Y-%m-%d %H:%M:%S"), "status": "skipped", "reason": repr(exc)}
        write_json(PACKAGE_ROOT / "r" / "local_pytorch_pth_latency_baseline_skipped.json", skip)
        print("Skipped local PyTorch baseline:", repr(exc))
        return None

local_pytorch_baseline = benchmark_local_pytorch_pth_forward()

local PyTorch latency:   0%|          | 0/40 [00:00<?, ?it/s]

{
  "created_at": "2026-05-10 11:23:15",
  "source": "local PyTorch best.pth CPU forward baseline",
  "frames": 40,
  "latency_ms": {
    "preprocess": {
      "mean": 2.5858050000351795,
      "p50": 2.5384000000485685,
      "p90": 3.1503100028203335,
      "p95": 3.2952850006040526,
      "max": 4.168499999650521
    },
    "pytorch_forward": {
      "mean": 107.00663249972422,
      "p50": 96.60819999953674,
      "p90": 138.98807999794371,
      "p95": 155.26973000269206,
      "max": 195.3529000020353
    },
    "pipeline": {
      "mean": 109.5924374997594,
      "p50": 98.75059999831137,
      "p90": 142.1778899992205,
      "p95": 157.6091700026154,
      "max": 197.93610000124318
    },
    "fps_pipeline_mean": 9.124717205073528
  },
  "note": "Reference only; Pi deployment uses ONNX Runtime.",
  "csv": "~\\02_Projects\\University\\26-1_EmbeddedArtificialSystemOptimization\\10_experiments\\12_clrkdnet_supervised_rebuild\\review_outputs\\10_pi_runtime_latency_sequence_validati

## Field3 Offline Replay Video

Pi에서 이 영상이 만들어지면, Pi의 ONNX + decoder + steering 전체 pipeline으로 field3 sequence를 재생한 것이다.

In [9]:
def draw_overlay_frame(bgr, lanes, row, latency_ms=None):
    img = bgr.copy()
    for lane in lanes:
        pts = np.asarray(lane["points"], dtype=np.int32)
        if len(pts) >= 2:
            cv2.polylines(img, [pts.reshape(-1, 1, 2)], False, (0, 255, 120), 3, cv2.LINE_AA)
    center_x = float(row.get("smoothed_center_x", IMAGE_CENTER_X))
    heading = float(row.get("smoothed_heading", 0.0))
    steer = float(row.get("steer_norm", 0.0))
    y0, y1 = RAW_H - 30, int((RAW_H - 1) * 0.78)
    x0 = int(IMAGE_CENTER_X)
    x1 = int(np.clip(center_x + heading * 220.0, 0, RAW_W - 1))
    cv2.line(img, (x0, y0), (x1, y1), (0, 165, 255), 4, cv2.LINE_AA)
    steer_x = int(np.clip(IMAGE_CENTER_X + steer * 420.0, 0, RAW_W - 1))
    cv2.arrowedLine(img, (x0, y0 - 35), (steer_x, y1 - 40), (255, 0, 255), 4, cv2.LINE_AA, tipLength=0.18)
    text = f"{int(row.get('order', 0)):04d} mode={row.get('effective_mode', '')} steer={steer:+.3f}"
    if latency_ms is not None:
        text += f" pipe={latency_ms:.1f}ms"
    cv2.rectangle(img, (18, 18), (820, 78), (0, 0, 0), -1)
    cv2.putText(img, text, (32, 58), cv2.FONT_HERSHEY_SIMPLEX, 0.85, (255, 255, 255), 2, cv2.LINE_AA)
    return img

def write_sequence_video(verify_result):
    if not WRITE_SEQUENCE_VIDEO or verify_result is None:
        return
    records = sorted([r for r in verify_result["records"] if r["role"] == "sequence"], key=lambda r: int(r["order"]))
    steer_by_key = {r["key"]: r for r in verify_result["steer_df"].to_dict("records")}
    lat_by_key = {r["key"]: r for r in verify_result["latency_df"].to_dict("records")}
    lanes_by_key = verify_result["seq_lanes_by_key"]
    video_dir = PACKAGE_ROOT / ("v/pi" if IS_PI else "v/local")
    video_dir.mkdir(parents=True, exist_ok=True)
    video_path = video_dir / "field3_runtime_replay.mp4"
    frame_dir = video_dir / "frames"
    frame_dir.mkdir(parents=True, exist_ok=True)
    writer = None
    for idx, rec in enumerate(records):
        bgr = imread_bgr(PACKAGE_ROOT / rec["image_rel"])
        row = steer_by_key[rec["key"]]
        lat = lat_by_key[rec["key"]]
        pipeline_ms = lat["preprocess_ms"] + lat["inference_ms"] + lat["decode_ms"] + lat["steering_ms"]
        frame = draw_overlay_frame(bgr, lanes_by_key[rec["key"]], row, pipeline_ms)
        if writer is None:
            h, w = frame.shape[:2]
            writer = cv2.VideoWriter(fs_path(video_path), cv2.VideoWriter_fourcc(*"mp4v"), 12.0, (w, h))
        writer.write(frame)
        if idx % 30 == 0:
            imwrite_bgr(frame_dir / f"frame_{idx:04d}.jpg", frame)
    if writer is not None:
        writer.release()
    print("video:", video_path)
    print("preview frames:", frame_dir)

write_sequence_video(verify_result)

video: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild\review_outputs\10_pi_runtime_latency_sequence_validation_v1\pkg\v\local\field3_runtime_replay.mp4
preview frames: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild\review_outputs\10_pi_runtime_latency_sequence_validation_v1\pkg\v\local\frames


In [10]:
if verify_result is not None:
    report_path = verify_result["out_dir"] / "runtime_validation_report.json"
    print("report:", report_path)
    print("overall_pass:", verify_result["summary"]["overall_pass"])
    print("pipeline_no_disk:", verify_result["summary"]["latency_ms"]["pipeline_no_disk"])
    print("fps_no_disk_mean:", verify_result["summary"]["latency_ms"]["fps_no_disk_mean"])
    print("package to copy to Pi:", PACKAGE_ROOT)

report: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild\review_outputs\10_pi_runtime_latency_sequence_validation_v1\pkg\out_local\runtime_validation_report.json
overall_pass: True
pipeline_no_disk: {'mean': 58.42129438198672, 'p50': 47.75760000120499, 'p90': 95.92435000013211, 'p95': 109.62422499960667, 'max': 240.77810000017053}
fps_no_disk_mean: 17.11704628558066
package to copy to Pi: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild\review_outputs\10_pi_runtime_latency_sequence_validation_v1\pkg


## Pi로 복사할 것

Local에서 Run All이 끝나면 아래 폴더 전체를 Pi로 복사한다.

`review_outputs/10_pi_runtime_latency_sequence_validation_v1/pkg/`

Pi에서는 이 `pkg/` 폴더 안에서 노트북을 열고 Run All 한다.

권장 Pi 패키지:

```bash
pip install onnxruntime numpy pandas opencv-python scipy tqdm
```

필수 확인:

- `m/*.onnx`
- `m/*.onnx.data`
- `c/*.json`
- `i/*.jpg`
- `r/ref_raw.npz`
- `r/ref_decoded.jsonl`
- `r/ref_steer_seq.csv`
- `t/records_manifest.csv`

10번이 통과하면 12번 실험 폴더는 ONNX 모델과 Pi runtime interface 검증까지 완료한 상태다. 실제 카메라/모터/표지판 state machine 통합은 별도 실주행 실험 폴더에서 진행한다.

## Pi 실행 결과 요약

이 셀은 Pi에서 실행한 `pkg` 전체를 로컬의 `pkg_from_pi/`로 다시 가져온 뒤, `out_pi/runtime_validation_report.json`을 기준으로 정리한 결과다.

### 결론

- **정합성은 통과**: Pi에서도 ONNX raw output, 07 decoder, 08 steering 결과가 로컬 reference와 같은 의미의 결과를 낸다.
- **실시간성은 부족**: Pi의 병목은 decoder나 steering이 아니라 **ONNX inference 자체**다.
- 현재 Pi 속도는 오프라인 검증/영상 리뷰에는 충분하지만, 바로 실주행에 쓰기에는 느리다.

### 정합성 검증

| 항목 | Pi 결과 | 기준 | 판단 |
|---|---:|---:|---|
| overall_pass | `True` | `True` | 통과 |
| raw max abs diff | `0.004089` | `0.02` 이하 | 통과 |
| raw mean abs diff max | `0.00000984` | `0.001` 이하 | 통과 |
| decode lane count mismatch | `0` | `0` | 통과 |
| decode max pair distance | `0.000165 px` | `2 px` 이하 | 통과 |
| steering mode mismatch | `0` | `0` | 통과 |
| steering max diff | `0.00000018` | `0.002` 이하 | 통과 |

해석: Pi ARM 환경에서도 `.onnx + numpy decoder + numpy steering` 조합은 로컬 reference와 같은 결과를 낸다. 배포 인터페이스 정합성 자체는 확보됐다.

### Latency 비교

| 항목 | Local ONNX | Pi ONNX | 해석 |
|---|---:|---:|---|
| pipeline no disk mean | `58.42 ms` | `343.46 ms` | Pi가 약 `5.88x` 느림 |
| pipeline no disk p95 | `109.62 ms` | `381.00 ms` | tail latency도 큼 |
| fps no disk mean | `17.12 fps` | `2.91 fps` | Pi는 약 3 fps |
| inference mean | `54.51 ms` | `339.70 ms` | 병목은 inference |
| preprocess mean | `3.18 ms` | `2.63 ms` | 충분히 작음 |
| decode mean | `0.62 ms` | `0.94 ms` | 충분히 작음 |
| steering mean | `0.10 ms` | `0.19 ms` | 충분히 작음 |

Pi breakdown:

```text
read       10.46 ms
preprocess 2.63 ms
inference  339.70 ms
decode     0.94 ms
steering   0.19 ms
pipeline   343.46 ms (2.91 fps)
```

### 판단

실주행용으로는 보통 `pipeline_no_disk.mean < 100 ms` 정도가 최소 실험선이고, `50~70 ms`면 더 안정적이다. 현재 Pi는 평균 `343.46 ms`, p95 `381.00 ms`라서 **실시간 주행에는 느리다**.

중요한 점은 후처리 병목이 아니라는 것이다. `decode + steering`은 평균 `1.13 ms` 수준이고, 대부분의 시간은 `inference` 평균 `339.70 ms`에서 발생한다.

### 산출물 위치

- Pi에서 가져온 전체 패키지: `~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild\review_outputs\10_pi_runtime_latency_sequence_validation_v1\pkg_from_pi`
- Pi report: `~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild\review_outputs\10_pi_runtime_latency_sequence_validation_v1\pkg_from_pi\out_pi\runtime_validation_report.json`
- Pi latency CSV: `~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild\review_outputs\10_pi_runtime_latency_sequence_validation_v1\pkg_from_pi\out_pi\latency_breakdown.csv`
- Pi replay video: `~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild\review_outputs\10_pi_runtime_latency_sequence_validation_v1\pkg_from_pi\v\pi\field3_runtime_replay.mp4`
- Pi archive: `~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild\review_outputs\10_pi_runtime_latency_sequence_validation_v1\pkg_from_pi.tar.gz`

### 다음 행동

1. 이 결과를 12번 실험의 결론으로 보존한다.
2. 다음 실험은 모델 정확도 문제가 아니라 **runtime 최적화 문제**로 분리한다.
3. 우선순위는 다음 순서가 좋다.

```text
1. ONNX Runtime 옵션 추가 점검
2. ONNX dynamic quantization 적용
3. quantized ONNX에 대해 09/10 parity 재검증
4. Pi latency 재측정
5. 그래도 느리면 입력 크기 축소 또는 모델 축소 검토
```

요약하면, 12번 폴더는 **모델/decoder/steering/export/Pi 정합성 검증까지는 성공**했다. 남은 문제는 **Pi에서 ResNet18 CLRKDNet ONNX inference가 약 340 ms로 느리다**는 점이며, 이는 별도의 양자화/최적화 실험으로 넘겨야 한다.
